# Task pipelines

[Back to the introduction](ch01_demo.ipynb). Run in order. `LLM.pipeline(task, model=...)` loads one Transformers.js task pipeline: zero-shot classification, speech recognition or translation. Each section loads a small browser model, runs it, and releases it before the next one. `backend` and `dtype` work as in `LLM.create()`.

In [ ]:
from lmlite import LLM, SUPPORTED_TASKS

print(SUPPORTED_TASKS)

## Zero-shot classification

Pick the labels at call time; no fine-tuned classifier is needed. Scores rank the labels and sum to 1 unless `multi_label=True`.

In [ ]:
classifier = await LLM.pipeline("zero-shot-classification", model="mobilebert-uncased-mnli", backend="wasm")
result = await classifier(
    "This notebook runs in a browser",
    candidate_labels=["technology", "sports", "cooking"],
)
for label, score in zip(result["labels"], result["scores"]):
    print(f"{label}: {score:.3f}")
await classifier.dispose()

## Translation

Opus-MT models translate one fixed language pair (here English to French). Pass a list to translate several sentences.

In [ ]:
translator = await LLM.pipeline("translation", model="opus-mt-en-fr", backend="wasm")
print(await translator("JupyterLite runs Python in the browser.", max_new_tokens=64))
print(await translator(["Hello!", "How are you?"]))
await translator.dispose()

## Speech recognition

Pass an HTTP(S) URL, a path to a PCM WAV file (for example one uploaded to the file browser), WAV bytes, or mono samples at 16 kHz. Whisper processes 30-second windows; use `chunk_length_s=30` for longer recordings. Decoding in a browser kernel is slower than native inference. `IPython.display.Audio` plays the clip in the page.

In [ ]:
from IPython.display import Audio, display

audio_url = "https://huggingface.co/datasets/Xenova/transformers.js-docs/resolve/main/jfk.wav"
display(Audio(url=audio_url))

In [ ]:
transcriber = await LLM.pipeline("automatic-speech-recognition", model="whisper-tiny.en", backend="wasm")
print(await transcriber(audio_url))
await transcriber.dispose()

The inference pipelines are released; cached files are retained for subsequent runs. See the usage guide for task options and output formats.